# 04b — Social Media Charts (Pillow)
Publication-ready PNG charts using the @unwelcomedata brand palette.
All charts export to twitter_landscape (1600×900px) with watermark.

**Rendering engine:** Pillow (PIL) — pixel-level control, no browser dependency.

**Production Charts (posting order):**
1. Top 10 Causes: Female vs Male (side-by-side, per 100k)
2. Top 10 Causes: White vs Black (side-by-side, per 100k)
3. National Abortion Comparison (stacked male/female with gestation breakdown)
3b. Top 5 Causes: White Americans (stacked + abortion) — supplemental
3c. Top 5 Causes: Black Americans (stacked + abortion) — supplemental
4. Per-Capita: White vs Black side-by-side (rate per 100k, shared scale)

**Data source:** All chart data stored in DuckDB `chart_*` tables.
Edit data in `04-viz.ipynb`, then re-run the table creation script if needed.


In [ ]:
# ===================================================================
# SETUP
# ===================================================================

import sys
import os
from pathlib import Path

import duckdb
import yaml

# Find project root
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT / 'src'))
sys.path.insert(0, str(PROJECT.parent / 'shared'))

from chart_factory import render_chart

with open(PROJECT / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

# Connect to DuckDB
conn = duckdb.connect(str(PROJECT / 'data' / 'project.duckdb'))

# Create outputs/social directory
social_dir = PROJECT / 'outputs' / 'social'
social_dir.mkdir(parents=True, exist_ok=True)

# Load annotation metadata
annotations = conn.execute("""
  SELECT key, value FROM chart_annotations
""").df().set_index('key')['value'].to_dict()

print('\u2713 chart_factory loaded')
print(f'\u2713 DuckDB connected')
print(f'\u2713 Charts export to: {social_dir}')
print(f'\u2713 Annotations: {annotations}')

In [ ]:
# ===================================================================
# COLOR PALETTE — edit here to experiment, re-run to update all charts
# ===================================================================

# --- Sex chart colors (Charts 1, 2, 3, 4) ---
C_MALE = '#0A9396'          # Dark Cyan
C_FEMALE = '#EE9B00'        # Golden Orange
C_ABORTION = '#AE2012'      # Oxidized Iron (accent)
C_SUICIDE_HL = '#94D2BD'    # Suicide highlight on male panel

# --- Race chart colors (Chart 1b) ---
C_WHITE = '#023047'         # Deep Space Blue
C_BLACK = '#6C757D'         # Slate Grey
C_RACE_SUICIDE_HL = '#0A9396'   # Suicide highlight on white panel
C_RACE_HOMICIDE_HL = '#EE9B00'  # Homicide highlight on black panel

# --- Detail bar gradients (4 steps: lightest to darkest) ---
GRAD_SUICIDE_MALE = ['#4FB3AA', '#94D2BD','#BFD5B2','#E9D8A6']
GRAD_SUICIDE_RACE = ['#4FB3AA', '#94D2BD','#BFD5B2','#E9D8A6']
GRAD_HOMICIDE_RACE = ['#EE9B00','#DC8101', '#D37402', '#CA6702']

# --- In-bar text color ---
C_BAR_TEXT = '#023047'

print('\u2713 Palette loaded')
print(f'  Male: {C_MALE}  |  Female: {C_FEMALE}  |  Abortion: {C_ABORTION}')
print(f'  White: {C_WHITE}  |  Black: {C_BLACK}')

---

## Chart 1: Top 10 Causes of Death — Female vs Male (Side-by-Side)

Two horizontal bar panels: Female top 10 on the left, Male top 10 on the right.
Each panel sorted independently by rate per 100k. Suicide highlighted on male panel
with annotation showing its rank for women.

In [ ]:
render_chart({
    'type': 'side_by_side_bars',
    'db': conn,
    'cfg': cfg,
    'preset': 'twitter_landscape',

    # Data
    'table_left': 'chart_female_top10',
    'table_right': 'chart_male_top10',
    'category_col': 'cause',
    'value_col': 'rate_per_100k',
    'label_col': 'total_label',

    # Text
    'title': 'Top 10 Causes of Death: Female vs Male',
    'subtitle': 'Rate per 100,000 population',
    'left_title': 'Female',
    'right_title': 'Male',
    'source': 'CDC WONDER 2024',

    # Colors
    'left_color': C_FEMALE,
    'right_color': C_MALE,

    # Highlight
    'highlight_right': {
        'category': 'Suicide',
        'color': C_SUICIDE_HL,
        'annotation': f'#{annotations["suicide_female_rank"]} for women',
    },

    # Detail bar
    'detail_bar': {
        'table': 'chart_male_suicide_age',
        'title': 'Male suicide by age:',
        'gradient': GRAD_SUICIDE_MALE,
    },

    # Export
    'filename': '01_top10_causes_female_vs_male',
})

---
## Chart 2: Top 10 Causes of Death — White vs Black (Side-by-Side)
Same pattern but comparing races. Suicide highlighted on white panel,
homicide highlighted on black panel. Two detail bars: white suicide by age,
black homicide victims by offender race.


In [ ]:
render_chart({
    'type': 'side_by_side_bars',
    'db': conn,
    'cfg': cfg,
    'preset': 'twitter_landscape',

    # Data
    'table_left': 'chart_white_top10',
    'table_right': 'chart_black_top10',
    'category_col': 'cause',
    'value_col': 'rate_per_100k',
    'label_col': 'total_label',

    # Text
    'title': 'Top 10 Causes of Death: White vs Black',
    'subtitle': 'Rate per 100,000 population',
    'left_title': 'White',
    'right_title': 'Black',
    'source': 'CDC WONDER 2024 · FBI SHR 2024',

    # Colors
    'left_color': C_WHITE,
    'right_color': C_BLACK,

    # Highlights
    'highlight_left': {
        'category': 'Suicide',
        'color': C_RACE_SUICIDE_HL,
        'annotation': f'#{annotations["suicide_black_rank"]} for Black',
    },
    'highlight_right': {
        'category': 'Homicide',
        'color': C_RACE_HOMICIDE_HL,
        'annotation': f'#{annotations["homicide_white_rank"]} for White',
    },

    # Detail bars
    'detail_bars': [
        {
            'table': 'chart_white_suicide_age',
            'title': 'White suicide by age:',
            'gradient': GRAD_SUICIDE_RACE,
        },
        {
            'table': 'chart_black_homicide_offender',
            'title': 'Black homicide victims \u2014 offender race:',
            'gradient': GRAD_HOMICIDE_RACE,
        },
    ],

    # Export
    'filename': '02_top10_causes_white_vs_black',
})

---
## Chart 3: National Abortion Comparison
What if abortion were counted as a cause of death?
Stacked male/female bars for top 5 causes, with abortion as a solid accent bar.
Inner segments show gestation timing.


In [ ]:
render_chart({
    'type': 'stacked_bars',
    'db': conn,
    'cfg': cfg,
    'preset': 'twitter_landscape',

    # Data
    'table': 'chart_national_stacked',
    'category_col': 'cause',

    # Segments
    'segments': [
        {'value_col': 'male_deaths', 'label': 'Male', 'color': C_MALE, 'text_color': C_BAR_TEXT},
        {'value_col': 'female_deaths', 'label': 'Female', 'color': C_FEMALE, 'text_color': C_BAR_TEXT},
        {'value_col': 'abortion_deaths', 'label': 'Abortion', 'color': C_ABORTION, 'text_color': '#E9D8A6'},
    ],

    # Inner segments: gestation age dividers inside the Abortion bar
    'inner_segments': {
        'category': 'Abortion',
        'table': 'chart_abortion_gestation',
        'text_color': '#E9D8A6',
        'divider_color': '#fff5e6',
        'min_width_pct': 10.0,
    },

    # Text
    'title': 'What if abortion were counted as a cause of death?',
    'subtitle': 'Top 5 causes of death, by sex',
    'source': 'CDC WONDER 2024 · Guttmacher Institute 2024',

    # Export
    'filename': '03_abortion_comparison_national',
})

---
## Chart 3b: Top 5 Causes — White Americans (+ Abortion)
Same stacked format filtered to White Americans.


In [ ]:
render_chart({
    'type': 'stacked_bars',
    'db': conn,
    'cfg': cfg,
    'preset': 'twitter_landscape',

    # Data
    'table': 'chart_stacked_white',
    'category_col': 'cause',

    # Segments
    'segments': [
        {'value_col': 'male_deaths', 'label': 'Male', 'color': C_MALE, 'text_color': C_BAR_TEXT},
        {'value_col': 'female_deaths', 'label': 'Female', 'color': C_FEMALE, 'text_color': C_BAR_TEXT},
        {'value_col': 'abortion_deaths', 'label': 'Abortion', 'color': C_ABORTION, 'text_color': '#E9D8A6'},
    ],

    # Text
    'title': 'What if abortion were counted as a cause of death?',
    'subtitle': 'Top 5 causes among White Americans, by sex',
    'source': 'CDC WONDER 2024 · Guttmacher Institute 2024',

    # Export
    'filename': '03b_top5_causes_white',
})

---
## Chart 3c: Top 5 Causes — Black Americans (+ Abortion)
Same stacked format filtered to Black Americans.


In [ ]:
render_chart({
    'type': 'stacked_bars',
    'db': conn,
    'cfg': cfg,
    'preset': 'twitter_landscape',

    # Data
    'table': 'chart_stacked_black',
    'category_col': 'cause',

    # Segments
    'segments': [
        {'value_col': 'male_deaths', 'label': 'Male', 'color': C_MALE, 'text_color': C_BAR_TEXT},
        {'value_col': 'female_deaths', 'label': 'Female', 'color': C_FEMALE, 'text_color': C_BAR_TEXT},
        {'value_col': 'abortion_deaths', 'label': 'Abortion', 'color': C_ABORTION, 'text_color': '#E9D8A6'},
    ],

    # Text
    'title': 'What if abortion were counted as a cause of death?',
    'subtitle': 'Top 5 causes among Black Americans, by sex',
    'source': 'CDC WONDER 2024 · Guttmacher Institute 2024',

    # Export
    'filename': '03c_top5_causes_black',
})

---
## Chart 4: Per-Capita Comparison — White vs Black
Same causes, expressed as a rate per 100,000 population.
Denominator includes aborted (if not aborted, they would be counted as population).
This allows direct cross-race comparison on a shared scale.


In [ ]:
render_chart({
    'type': 'side_by_side_bars',
    'db': conn,
    'cfg': cfg,
    'preset': 'twitter_landscape',
    # Data
    'table_left': 'chart_white_percapita',
    'table_right': 'chart_black_percapita',
    'category_col': 'cause',
    'value_col': 'rate_per_100k',
    'label_col': 'total_label',
    # Text
    'title': 'Abortion as a cause of death: White vs Black',
    'subtitle': 'Rate per 100,000 population',
    'left_title': 'White',
    'right_title': 'Black',
    'source': 'CDC WONDER 2024 · Guttmacher Institute 2024',
    # Colors
    'left_color': C_WHITE,
    'right_color': C_BLACK,
    # Highlight abortion bars
    'highlight_left': {'category': 'Abortion', 'color': C_ABORTION},
    'highlight_right': {'category': 'Abortion', 'color': C_ABORTION},
    # Export
    'filename': '04_percapita_white_vs_black',
})


---

## Summary & Cleanup

In [ ]:
conn.close()

# Verify all exports
pngs = sorted(social_dir.glob('*.png'))
print('=== ALL CHARTS COMPLETE ===')
print(f'\n\u2713 Generated {len(pngs)} publication-ready charts:')
for png in pngs:
    size_kb = png.stat().st_size / 1024
    print(f'  \u2022 {png.name} ({size_kb:.0f} KB)')

print('\nAll charts are twitter_landscape (1600\u00d7900px) with @unwelcomedata watermark.')
print('Ready for social media posting!')